## Building A Better RAG System

### Dev Guide 
### AI Servies request for support

## Outline:
## Starting with a basic RAG system
## Evaluating a RAG system
## Cleaning strategies
## Chunking strategies
## Labelling stragegies
## Choice of embedding model
## Reranker models
## Building train/test data for RAG
## Fine-tuning an embedding model
## General solutions for RAG problems

In [ ]:
import os
import math
from typing import List
from PyPDF2 import PdfReader #3.0.1

import httpx                # v0.28.1 HTTP client with TLS verification
import openai               # v2.3.0 Compatible client for Mission Assist
import chromadb             # v1.3.5 Vector store

# Cleaning
import ftfy  #6.3.1
import textacy  #0.13.0
import trafilatura  #1.5.0
import datasketch  #1.10.0
import blingfire  #0.1.8
import symspellpy  #6.9.0

# Chunking
from typing import List, Any
from langchain.text_splitter import RecursiveCharacterTextSplitter 
#This is done in 3 imports
#langchain==0.3.27
#langchain_openai==0.3.35
#langchain_community==0.3.31
from llama_index.core.node_parser import SemanticSplitterNodeParser ##llama index 0.14.22
from llama_index.core.embeddings import BaseEmbedding
from llama_index.core import Document

# Labelling
#from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field # v2.12.0

#Generate train/test and fine-tune
import pandas as pd 
import pickle
import os
import pandas as pd
import json
import re
#pip install sentence_transformers[train]==5.6.0
from sentence_transformers import InputExample, SentenceTransformer, SentenceTransformerTrainer, losses, CrossEncoder, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import util
#You are going to need to find the version of torch that you need
import torch
import numpy as np
import random
from datasets import Dataset
import concurrent.futures
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

: 

In [ ]:
root_path = os.getcwd()+'/'
ca_path=root_path+'CA03_Base64_FullRootChain.pem'
api_key=''

## Extract text from PDFs for training purposes. You can use your own PDFs.

In [ ]:
def extract_text_from_pdfs(folder_path):
    # Initialize lists to store results
    texts = []
    sources = []

    # Ensure the directory exists
    if not os.path.exists(folder_path):
        print(f"Error: The folder {folder_path} does not exist.")
        return texts, sources

    # List all files in the directory and filter for PDFs
    files = os.listdir(folder_path)
    pdf_files = [f for f in files if f.lower().endswith('.pdf')]

    if not pdf_files:
        print("No PDF files found in the specified directory.")
        return texts, sources

    for pdf_file in pdf_files:
        file_path = os.path.join(folder_path, pdf_file)
        
        try:
            reader = PdfReader(file_path)
            full_text = ""
            
            # Extract text from every page
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    full_text += extracted + "\n"
            
            # Append the filename and the extracted text to the lists
            sources.append(pdf_file)
            texts.append(full_text.strip())
            
        except Exception as e:
            print(f"Could not read file {pdf_file}: {e}")

    return texts, sources

target_folder = root_path+"example_pdfs/"
texts, sources = extract_text_from_pdfs(target_folder)

In [ ]:
for i in range(0,len(texts)):
    print(f"{len(texts[i])} characters for text {sources[i]}")

## This example is identical to the API dev guide sample project except for text extraction

In [ ]:
class MissionAssistProcessor:
    """
    Handles document loading, chunking, embedding, storage, and LLM-based QA.
    """

    def __init__(
        self,
        api_key: str,
        ca_path: str,
        embed_service: str = "api-snowflake",   
        llm_service: str = "api-gptoss",       
        chroma_path: str = "./chroma_db",
        collection_name: str = "mission_assist_docs",
    ):
        # ---- HTTP client ----
        self.http_client = httpx.Client(verify=ca_path)
        self.headers = {"apikey": api_key}

        # ---- Embedding client ----
        embed_base = f"https://dev.api.com/{embed_service}/v1"
        self.embed_client = openai.OpenAI(
            api_key=api_key,
            base_url=embed_base,
            default_headers=self.headers,
            http_client=self.http_client,
        )
        self.embed_model = self.embed_client.models.list().data[0].id

        # ---- LLM client ----
        llm_base = f"https://dev.api.us.com/{llm_service}/v1"
        self.llm_client = openai.OpenAI(
            api_key=api_key,
            base_url=llm_base,
            default_headers=self.headers,
            http_client=self.http_client,
        )
        self.llm_model_name = self.llm_client.models.list().data[0].id

        # ---- ChromaDB ----
        self.chroma_client = chromadb.PersistentClient(path=chroma_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name
        )

        # ---- Chunking parameters ----
        self.chunk_size = 500      
        self.step = 250            
        self.min_chunk_len = 200   

    def _chunk_text(self, text: str) -> List[str]:
        length = len(text)
        if length <= self.chunk_size:
            return [text]

        n_orig = math.ceil((length - self.chunk_size) / self.step) + 1
        n_chunks = max(1, round(n_orig))
        adjusted_chunk_size = length - (n_chunks - 1) * self.step

        while adjusted_chunk_size < self.min_chunk_len and n_chunks > 1:
            n_chunks -= 1
            adjusted_chunk_size = length - (n_chunks - 1) * self.step

        chunks: List[str] = []
        start = 0
        for _ in range(n_chunks):
            end = start + adjusted_chunk_size
            chunks.append(text[start:end])
            start += self.step
        return chunks

    def _embed(self, texts: List[str]) -> List[List[float]]:
        response = self.embed_client.embeddings.create(
            input=texts,
            model=self.embed_model,
        )
        return [item.embedding for item in response.data]

    # ------------------------------------------------------------------
    # NEW METHOD: Ingest using the pre-extracted texts and sources lists
    # ------------------------------------------------------------------
    def ingest_preprocessed_data(self, texts: List[str], sources: List[str]) -> None:
        """
        Takes lists of extracted text and corresponding filenames.
        """
        if len(texts) != len(sources):
            raise ValueError("The length of texts and sources lists must be identical.")

        for idx, (content, source_name) in enumerate(zip(texts, sources)):
            if not content:
                print(f"[WARN] No content for {source_name}; skipping.")
                continue

            chunks = self._chunk_text(content)
            embeddings = self._embed(chunks)

            # Use the source filename for ID generation
            ids = [f"{source_name}_chunk{i}" for i in range(len(chunks))]

            self.collection.add(
                documents=chunks,
                embeddings=embeddings,
                ids=ids,
            )
            print(f"[INGEST] {source_name}: {len(chunks)} chunk(s) stored (doc {idx + 1}/{len(texts)})")

    def _retrieve_chunks(self, query_text: str, top_k: int = 5) -> List[str]:
        query_emb = self._embed([query_text])[0]
        result = self.collection.query(
            query_embeddings=[query_emb],
            n_results=top_k,
        )
        return result["documents"][0] if result["documents"] else []

    def _call_llm(self, prompt: str) -> str:
        response = self.llm_client.chat.completions.create(
            model=self.llm_model_name,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a helpful assistant. Answer the question "
                        "using only the provided context. If the context does not "
                        "contain enough information, respond with \"I don't have enough "
                        "information to answer that.\""
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.7,
        )
        return response.choices[0].message.content

    def ask_question(self, question: str, top_k: int = 5) -> str:
        relevant = self._retrieve_chunks(question, top_k=top_k)
        if not relevant:
            return "No relevant context found."

        context = "\n---\n".join(relevant)
        prompt = f"Context:\n{context}\n\nQuestion: {question}"
        return self._call_llm(prompt)

def ask_and_print(processor: MissionAssistProcessor, question: str, top_k: int = 5) -> None:
    answer = processor.ask_question(question, top_k=top_k)
    print("\n❓ Question:", question)
    print("\n🤖 Answer:\n", answer)

In [ ]:
processor = MissionAssistProcessor(
    api_key=api_key,
    ca_path=ca_path
)

# Use the new method instead of .ingest()
processor.ingest_preprocessed_data(texts, sources)

In [ ]:
ask_and_print(processor, "What is AE?")

# How can we evaluate this system? 
### There are many things we can do to improve RAG. Without a concrete evaluation metric we are just guessing.

## Wikipedia page on metrics -- section on offline metrics

### https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)

## Some questions to consider for your RAG system when choosing evaluation metric(s)
### Is it sufficient to identify the correct document? Chunk? Sentence?
### For each 'correct' doucment, are there potential alternate documents that also have the same information?
### Are there common 'types' of questions being asked? If yes, could we score independently?
### Could you produce 100 questions that your users will ask commonly? Could you provide an answer document for each?
### Could you use of an LLM to produce questions about a document and then verify the document as a source for your domain?

## Cleaning Strategies

In [ ]:
import ftfy
import textacy
import textacy.preprocessing 
import trafilatura
from symspellpy import SymSpell, Verbosity
import blingfire as bf
from datasketch import MinHash
import numpy as np
import re

# --- SETUP FOR DEMO ---
raw_dirty_text = """
Quarterly Update: AI Integration

This is a test of the system.   The quick brown fox jumps over the lzy dog. 
The quick brown fox jumps over the lzy dog. (Note: this is a near-duplicate sentence).

    We are   excited to announce our new project.    It is truly amazng! 
Please reach out to us at support@example.com for more info.   

Encoding test: The price is 100&euro; and the date is 2023â\x80\x9410-12.
"""

sample_html = f"""
<html>
<body>
    <nav>Home | About | Contact</nav>
    <div class="content">{raw_dirty_text}</div>
</body>
</html>
"""

print("=== DATA CLEANING PIPELINE DEMONSTRATION ===\n")
print("--- ORIGINAL INPUT (RAW) ---")
print(raw_dirty_text)
print("-" * 50)

# 1. Trafilatura: Boilerplate Removal
extracted_text = trafilatura.extract(sample_html)
print(f"Step 1: Trafilatura (HTML -> Main Text)\nGoal: Remove HTML tags and boilerplate (nav/footer) to isolate core content.\nResult:\n{extracted_text}\n")
print("-" * 50)

# 2. ftfy: Fixing Encoding
fixed_text = ftfy.fix_text(raw_dirty_text)
print(f"Step 2: ftfy (Fixing Mojibake/Encoding)\nGoal: Repair broken encoding (Mojibake) and normalize HTML entities (e.g., &euro; -> €).\nResult:\n{fixed_text}\n")
print("-" * 50)

# 3. SymSpell: Spelling Correction
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
manual_dictionary = {"amazing": 100, "lazy": 100}
for term, count in manual_dictionary.items():
    sym_spell.create_dictionary_entry(term, count)

def correct_spelling_preserve_space(text):
    """
    Custom correction that fixes spelling but PRESERVES 
    whitespace so textacy can still demonstrate its value.
    """
    if not text: return ""
    # Use regex to find words but keep the whitespace exactly as is
    def replace_word(match):
        word = match.group(0)
        clean_w = re.sub(r'[^\w]', '', word)
        suggestions = sym_spell.lookup(clean_w, Verbosity.CLOSEST, max_edit_distance=2)
        if suggestions:
            return word.replace(clean_w, suggestions[0].term)
        return word

    return re.sub(r'\b\w+\b', replace_word, text)

spelled_text = correct_spelling_preserve_space(fixed_text)
print(f"Step 3: SymSpell (Spelling Correction)\nGoal: Correct typographical errors (e.g., 'lzy' -> 'lazy') using a frequency dictionary.\nResult:\n{spelled_text}\n")
print("-" * 50)

# 4. textacy: Whitespace Normalization
cleaned_text = textacy.preprocessing.normalize.whitespace(spelled_text)
print(f"Step 4: textacy (Whitespace Normalization)\nGoal: Collapse redundant whitespace, tabs, and newlines into a standardized single-space format.\nResult:\n{cleaned_text}\n")
print("-" * 50)

# 5. blingfire/re: Pattern Masking
try:
    masked_text = bf.replace(cleaned_text, r"[\w\.-]+@[\w\.-]+", "[EMAIL_REDACTED]")
except AttributeError:
    masked_text = re.sub(r"[\w\.-]+@[\w\.-]+", "[EMAIL_REDACTED]", cleaned_text)

print(f"Step 5: blingfire/re (PII Masking)\nGoal: Identify and redact Personally Identifiable Information (PII) using high-performance regex.\nResult:\n{masked_text}\n")
print("-" * 50)

# 6. datasketch: Near-Duplicate Detection
def get_minhash(text):
    m = MinHash(num_perm=128)
    if not text: return m
    for word in text.lower().split():
        m.update(word.encode('utf8'))
    return m

m1 = get_minhash(raw_dirty_text)
m2 = get_minhash(masked_text)
similarity = m1.jaccard(m2)
print(f"Step 6: datasketch (Similarity Analysis)\nOriginal vs Cleaned Jaccard Score: {similarity:.4f}")
print("\nInterpretation: The high score proves that the cleaning process removed noise (formatting/typos) without altering the semantic meaning of the document.")

## Chunking Stragegies

In [ ]:
from typing import List, Any
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from llama_index.core.node_parser import SemanticSplitterNodeParser ##llama index 0.14.22
from llama_index.core.embeddings import BaseEmbedding
from llama_index.core import Document

header = {"apikey": api_key} 
url = 'api-snowflake'

client = openai.OpenAI(
    api_key=api_key, 
    base_url=f'https://dev.api.com/{url}/v1/',
    http_client=httpx.Client(verify=ca_path),
    default_headers=header
)

# Get the model ID automatically from the list
models_list = client.models.list()
model_id = models_list.data[0].id

# Create a Custom Embedding Class for LlamaIndex
class Embedding(BaseEmbedding):
    # Define these as Pydantic fields so they are handled correctly by the base class
    client: Any
    model_id: str

    # --- Synchronous Methods ---
    def _get_text_embedding(self, text: str) -> List[float]:
        response = self.client.embeddings.create(input=[text], model=self.model_id)
        return response.data[0].embedding

    def _get_query_embedding(self, query: str) -> List[float]:
        return self._get_text_embedding(query)

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        response = self.client.embeddings.create(input=texts, model=self.model_id)
        return [item.embedding for item in response.data]

    # --- Asynchronous Methods ---
    async def _aget_query_embedding(self, query: str) -> List[float]:
        return self._get_query_embedding(query)

    async def _aget_text_embedding(self, text: str) -> List[float]:
        return self._get_text_embedding(text)

    async def _aget_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return self._get_text_embeddings(texts)

# Initialize using keyword arguments (Pydantic style)
embed_model = Embedding(client=client, model_id=model_id)

# ==========================================
# INPUT TEXT
# ==========================================
text_content = """
Artificial Intelligence is transforming the modern workplace. Large Language Models like GPT-4 
and Claude are enabling developers to write code faster and helping analysts summarize 
massive datasets in seconds. The integration of AI into daily workflows is no longer 
a luxury but a necessity for competitive edge.

On a completely different note, let's talk about the art of French pastry. 
The secret to a perfect macaron lies in the 'macaronage' process, where the batter 
is folded just enough to remove air bubbles without deflating the mixture. 
Precision in temperature and humidity is critical for the signature smooth top.

Moving far beyond Earth, the James Webb Space Telescope is uncovering the secrets of the early universe. 
By capturing infrared light, it can peer through cosmic dust clouds to see the first stars 
forming. This allows astronomers to look back billions of years into the past, 
effectively using the telescope as a time machine.
"""

print("=== CHUNKING STRATEGY COMPARISON (EMBEDDINGS) ===\n")

# ------------------------------------------------------------------------------
# STRATEGY 1: Naive (Fixed-Size)
# ------------------------------------------------------------------------------
def naive_chunking(text, size=150):
    return [text[i:i+size] for i in range(0, len(text), size)]

naive_chunks = naive_chunking(text_content)

print("STRATEGY 1: Naive (Fixed-Size)")
print("Goal: Split strictly by character count.")
for i, chunk in enumerate(naive_chunks):
    print(f"Chunk {i+1}: {chunk.replace(chr(10), ' ')}...") 
print("-" * 60)

# ------------------------------------------------------------------------------
# STRATEGY 2: Structural (Recursive)
# ------------------------------------------------------------------------------
structural_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)
structural_chunks = structural_splitter.split_text(text_content)

print("STRATEGY 2: Structural (Recursive)")
print("Goal: Respect boundaries (paragraphs/sentences) while staying near a size limit.")
for i, chunk in enumerate(structural_chunks):
    print(f"Chunk {i+1}: {chunk.replace(chr(10), ' ')}...")
print("-" * 60)

# ------------------------------------------------------------------------------
# STRATEGY 3: Semantic (Embedding-Based)
# ------------------------------------------------------------------------------
print("STRATEGY 3: Semantic (Embedding-Based)")
print("Goal: Split only when the subject matter actually changes.")

semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percent_threshold=40, 
    embed_model=embed_model 
)

node_objects = semantic_splitter.get_nodes_from_documents([Document(text=text_content)])
semantic_chunks = [node.get_content() for node in node_objects]

for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i+1}: {chunk.replace(chr(10), ' ')}...")
print("-" * 60)

# ==========================================
# FINAL COMPARISON SUMMARY
# ==========================================
print("\n=== FINAL ANALYSIS ===")
print(f"Naive Chunks:     {len(naive_chunks)}")
print(f"Structural Chunks: {len(structural_chunks)}")
print(f"Semantic Chunks:   {len(semantic_chunks)}")
print("\nObservations:")
print("1. Naive: Likely cut sentences in half (visible in output).")
print("2. Structural: Kept sentences whole but ignored the topic change.")
print("3. Semantic: Used embeddings to identify that AI, Cooking, and Space are different topics.")

## Labelling strategies

### Do your documents have a structure?
### Is there an existing labelling system?
### Do you have a list of parts/terms/projects? If you have a definition for each could an LLM help tag?
### Graph Database vs Vector Database vs Relational Database

In [ ]:
from langchain_openai import ChatOpenAI
from typing import Optional
from pydantic import BaseModel, Field
import httpx
import json

model_name='/genai/Llama-3.3-70b'
url='api-llama'

header = {"apikey": api_key} 

class MetadataLabel(BaseModel):
    """Metadata labels for a technical document string."""

    category: str = Field(
        description="The primary category of the text. Options: 'Technical Specification', 'Policy/Governance', 'Financial', or 'General Inquiry'."
    )
    sentiment: str = Field(
        description="The tone of the text: 'Urgent', 'Informational', or 'Critical'."
    )
    confidence_score: float = Field(
        description="A value between 0.0 and 1.0 representing the model's confidence in this labeling."
    )
    summary: Optional[str] = Field(
        default=None, 
        description="A concise one-sentence summary of the labeled content."
    )

llm = ChatOpenAI(
    api_key=api_key,
    base_url=f'https://dev.api.com/{url}/v1', 
    default_headers=header,
    http_client=httpx.Client(verify=ca_path),
    response_format=MetadataLabel,
    temperature=0.1, # Lower temperature is generally better for labeling tasks
    model=model_name
    )

# Example string to be labeled
text_to_label = "The updated DFARS 252.204-7012 requirement mandates that all covered contractor information be protected according to NIST SP 800-171 standards immediately."

messages=[
    {
        "role": "user",
        "content": f"Please label the following text with the required metadata: {text_to_label}",
    }
]

response=llm.invoke(messages)
print(json.loads(response.content))

## Choice of Embedding Model
### https://huggingface.co/spaces/mteb/leaderboard

## Reranker Model
### https://sbert.net/docs/cross_encoder/usage/usage.html

In [ ]:
header = {"apikey": api_key}
RERANKER_MODEL_URL = "https://dev.api.us.com/api-nemotron-rerank/v1"
http_client = httpx.Client(verify=ca_path)
client = openai.OpenAI(base_url=RERANKER_MODEL_URL, api_key=api_key, http_client=http_client, default_headers=header)
print(client.models.list())

In [ ]:
def demonstrate_reranking():
    query = "What are the guidelines for handling CUI data?"

    retrieved_documents = [
        "The company cafeteria is open from 11:00 AM to 2:00 PM daily.",
        "CUI (Controlled Unclassified Information) must be encrypted at rest and in transit.",
        "Employee parking is available in the North and South lots.",
        "All personnel handling CUI must complete the annual security awareness training.",
        "The project timeline for the new wing is estimated at 18 months."
    ]

    print(f"Query: {query}")
    print("\n--- Initial Retrieval (Unsorted/Low Precision) ---")
    for i, doc in enumerate(retrieved_documents):
        print(f"Doc {i}: {doc}")

    try:
        response = http_client.post(
            f"{RERANKER_MODEL_URL}/rerank", 
            headers=header,
            json={
                "query": query,
                "documents": retrieved_documents,
                "top_n": 3
            }
        )
        response.raise_for_status()
        results = response.json() 
        
        # DEBUG: Print the raw response so you can see the actual keys being used
        # print(f"DEBUG RAW RESPONSE: {json.dumps(results, indent=2)}")

        print("\n--- After Reranking (High Precision) ---")
        
        # We look for the list of results. 
        # If the API returns the list directly instead of inside a "results" key, we handle that.
        data_list = results.get("results", results) if isinstance(results, dict) else results

        for item in data_list:
            # Use .get() to avoid KeyError. 
            # We try common keys: 'score', 'relevance_score', 'value'
            idx = item.get('index')
            score = item.get('score') or item.get('relevance_score') or item.get('value', 0.0)
            
            if idx is not None:
                print(f"Score: {score:.4f} | Doc {idx}: {retrieved_documents[idx]}")
            else:
                print(f"Could not find index in item: {item}")

    except Exception as e:
        print(f"Error during reranking: {e}")
        # If it fails, print the response body to help diagnose the schema
        if 'response' in locals():
            print(f"Server Response: {response.text}")

demonstrate_reranking()

### What does it mean for an embedding model to rank responses vs. a cross encoder?

### A reranker beats fine-tuning an embedding model in some cases

## Building train/test data for RAG

In [ ]:
torch.cuda.is_available()
#make sure cuda is available. You need a version of torch that matches your nvidia driver (nvidia-smi)

### Load data. We are going to create two lists -- texts and source. texts will be the content chunked into pieces and the source will be a metadata label for the chunk. In this case we label each chunk with the title of the document it came from.

In [ ]:
#We used this same function eariler. This starts the same way.
def extract_text_from_pdfs(folder_path):
    # Initialize lists to store results
    texts = []
    sources = []

    # Ensure the directory exists
    if not os.path.exists(folder_path):
        print(f"Error: The folder {folder_path} does not exist.")
        return texts, sources

    # List all files in the directory and filter for PDFs
    files = os.listdir(folder_path)
    pdf_files = [f for f in files if f.lower().endswith('.pdf')]

    if not pdf_files:
        print("No PDF files found in the specified directory.")
        return texts, sources

    for pdf_file in pdf_files:
        file_path = os.path.join(folder_path, pdf_file)
        
        try:
            reader = PdfReader(file_path)
            full_text = ""
            
            # Extract text from every page
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    full_text += extracted + "\n"
            
            # Append the filename and the extracted text to the lists
            sources.append(pdf_file)
            texts.append(full_text.strip())
            
        except Exception as e:
            print(f"Could not read file {pdf_file}: {e}")

    return texts, sources

target_folder = root_path+"example_pdfs/"
texts, sources = extract_text_from_pdfs(target_folder)


### The dataset we are going to build creates 3 questions per chunk. Google reccomends at least 10k of the training triplets we are going to create. We need to have some data for validation stats. We will generate 3 questions per chunk.
### We create connections to the models needed and then set up functions for dataset building. We start by vectorizing the chunks and uploading to a local chroma. Having this vectordb speed up negative mining.

In [ ]:
model_name = '/genai/Gemma-4-31B-IT'
url = 'api-gemma-4-31B'

RERANKER_MODEL_URL = "https://dev.api.com/api-nemotron-rerank/v1"

# Initialize Client
header = {"apikey": api_key} 
llm = OpenAI(
    api_key=api_key,
    base_url=f'http://dev.api.com/{url}/v1', 
    default_headers=header
)

#create a local chroma with the vectors 
# --- Configuration ---
url = 'api-e5-mistral7b'
CHROMA_PATH = "/mnt/c/Users/luke.flattery/Downloads/embed_model/chroma_db"
COLLECTION_NAME = "document_embeddings"
BATCH_SIZE = 100 # Adjust based on model limits

# --- Client Setup ---
# OpenAI Client for Embeddings
client = openai.OpenAI(
    api_key=api_key, 
    base_url=f'http://dev.api.com/{url}/v1/',
    default_headers=header
)

# ChromaDB Local Client
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

# Get the model ID dynamically as per your snippet
models = client.models.list()
model_id = models.data[0].id

In [ ]:
def embed_and_store(texts: List[str], sources: List[str]):
    """
    Embeds text chunks and stores them in a local ChromaDB instance with source metadata.
    """
    # Ensure texts and sources are the same length
    if len(texts) != len(sources):
        raise ValueError("The length of texts and sources lists must be identical.")

    # Process in batches to avoid API timeouts or payload limits
    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i : i + BATCH_SIZE]
        batch_sources = sources[i : i + BATCH_SIZE]
        
        # 1. Generate Embeddings
        responses = client.embeddings.create(
            input=batch_texts,
            model=model_id,
        )
        
        # Extract embeddings from the response
        embeddings = [item.embedding for item in responses.data]
        
        # 2. Prepare ChromaDB requirements
        # IDs must be unique strings
        ids = [f"id_{j}" for j in range(i, i + len(batch_texts))]
        
        # Metadata must be a list of dictionaries
        metadatas = [{"source": src} for src in batch_sources]
        
        # 3. Add to Chroma
        collection.add(
            embeddings=embeddings,
            documents=batch_texts,
            metadatas=metadatas,
            ids=ids
        )
        
        print(f"Processed batch {i // BATCH_SIZE + 1}: {i + len(batch_texts)}/{len(texts)} items stored.")

#test query if you want to test the database
def query_local_db(query_text: str, n_results: int = 3):
    """
    Embeds the query and retrieves the most similar documents from ChromaDB.
    """
    print(f"\nQuerying: '{query_text}'")
    print("-" * 50)

    # 1. Embed the query text using the API
    response = client.embeddings.create(
        input=[query_text],
        model=model_id,
    )
    query_embedding = response.data[0].embedding

    # 2. Query ChromaDB
    # We pass the embedding directly using the 'query_embeddings' parameter
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    # 3. Parse and Print Results
    # Chroma returns lists of lists (because you can query multiple embeddings at once)
    for i in range(len(results['ids'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i]
        doc_id = results['ids'][0][i]
        
        print(f"Rank: {i+1}")
        print(f"ID: {doc_id}")
        print(f"Distance: {dist:.4f} (Lower is more similar)")
        print(f"Source: {meta.get('source', 'Unknown')}")
        print(f"Text: {doc}")
        print("-" * 50)


embed_and_store(texts, sources)
print("Successfully embedded and stored all documents.")

In [ ]:
def choose_related_chunk(query_text: str, id_integer: int):
    """
    Embeds the query, finds the 10 closest chunks, and randomly returns one 
    that does not match the provided id_integer.
    """
    # 1. Embed the query text using the API
    response = client.embeddings.create(
        input=[query_text],
        model=model_id,
    )
    query_embedding = response.data[0].embedding

    # 2. Query ChromaDB for the top 10 results
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10
    )
    # Extract IDs and Documents into flat lists
    # Chroma returns lists of lists; we take the first index [0]
    ids = results['ids'][0]
    documents = results['documents'][0]
    
    # 3. Create a pool of candidates that are NOT the forbidden ID
    forbidden_id = f"id_{id_integer}"
    candidates=[]
    for i in range(len(ids)):
        if ids[i] != forbidden_id:
            candidates.append(documents[i])
    # 4. Randomly pick one and return only the text
    if not candidates:
        return "No suitable related chunks found."

    return random.choice(candidates)
    
def generate_fine_tuning_questions(text, max_retries=3):
    """
    Generates exactly three high-quality questions from the provided text.
    Retries if the model fails to provide exactly three questions.
    """
    attempt = 0
    
    # Prompt designed to enforce strict formatting for easier parsing
    system_prompt = (
        """You are an expert data generation system for training embedding models.
Your task is to read the provided text and generate exactly 3 high-quality retrieval style questions that are strictly answerable from the text.
The questions will be use for embedding model fine-tuning so they must maximize semantic relevance, retrieval quality, and domain specificity.

Requirements:
Output exactly 3 questions
Output ONLY the questions
One question per line
Do not number the questions
Do not add explanations, prefixes, suffixes, bullets, or extra formatting.
Questions must be highly specific to the content of the text.
Questions should capture key entities, relationships, concepts, procedures or terminology.
Prefer questions that a domain expert would naturally ask
Avoid generic or vague questions.
Avoid questions that can be answered without the text.
Use diverse question styles and semantic angles.
If the text is technical, produce technically precise questions using the same terminology.
Questions should be concise but information dense.
Questions should optimize retrieval usefulness rather than conversational style.

Good Qualities:
Specific terminology
High information content
Strong grounding in the text
Different retrieval intents
Domain-aware phrasing

Bad qualities:
Generic summaries
Broad open-ended questions
Repetitive wording
Conversational fluff
Questions unrelated to the text

Return only the 3 questions."""
    )

    while attempt < max_retries:
        attempt += 1
        
        chat_response = llm.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text: {text}"},
            ],
            temperature=0.7 # Slightly higher temperature for better question variety
        )
        
        content = chat_response.choices[0].message.content.strip()
        # Split by newline and remove any empty strings resulting from extra whitespace
        questions = [q.strip() for q in content.split('\n') if q.strip()]
        
        # Validate that we have exactly 3 questions
        if len(questions) == 3:
            return questions
        
        print(f"Attempt {attempt} failed: Received {len(questions)} questions. Retrying...")

    print("Max retries reached. Could not generate exactly three questions.")
    return None

def rerank_score(query, chunk1, chunk2):
    """
    Sends a query and two document chunks to the reranker model 
    and returns the relevance scores for each.
    """
    # Combine chunks into a list for the API request
    documents = [chunk1, chunk2]
    
    try:
        response = requests.post(
            "https://dev.api.com/api-nemotron-rerank/v1/rerank",
            json={
                "model": "/genai/nvidiallama-nemotron-rerank-vl-1b-v2",
                "query": query,
                "documents": documents,
                "top_n": 2,
            },
            timeout=30,
            verify=False
        )
        response.raise_for_status()
        results = response.json()["results"]
        
        # The API returns results as a list of dictionaries containing 'index' and 'relevance_score'.
        # We map these back to the original order of chunk1 and chunk2.
        scores = {item["index"]: item["relevance_score"] for item in results}
        
        # Return the score for chunk1 (index 0) and chunk2 (index 1)
        return scores.get(0, 0.0), scores.get(1, 0.0)
        
    except:
        pass

def process_text(i, text):
    """Helper function to process a single text entry."""
    local_results = []
    try:
        questions = generate_fine_tuning_questions(text)
    except Exception:
        return []

    if len(questions) > 2:
        for q in questions:
            a = choose_related_chunk(q, i)
            score = rerank_score(q, text, a)
            if score:
                # Return as a tuple to maintain order: (q, pos, neg, pos_score, neg_score)
                local_results.append((q, text, a, score[0], score[1]))
    
    return local_results

#generate_fine_tuning_questions(texts[200])
#chosen_chunk = choose_related_chunk("what is SSP",2)


# Initialize lists
queries = []
positives = []
negatives = []
positives_scores = []
negatives_scores = []

# Use ThreadPoolExecutor for I/O bound tasks (API calls/DB lookups)
# max_workers can be adjusted based on your API rate limits
with ThreadPoolExecutor(max_workers=10) as executor:
    # Create a list of futures
    futures = [executor.submit(process_text, i, text) for i, text in enumerate(texts)]
    
    # Wrap the as_completed generator with tqdm for the progress bar
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Texts"):
        result_list = future.result()
        for res in result_list:
            queries.append(res[0])
            positives.append(res[1])
            negatives.append(res[2])
            positives_scores.append(res[3])
            negatives_scores.append(res[4])              


# 1. Group the lists into a dictionary
scores=[]
for i in range(0,len(positives_scores)):
    scores.append(positives_scores[i]-negatives_scores[i])
data_to_save = Dataset.from_dict({
    "text1": queries,
    "text2": positives,
    "text3": negatives,
    "label": scores,
})

# 2. Save to a file
file_path = "/mnt/c/Users/luke.flattery/Downloads/embed_model/fine_tuning_data.pkl"
with open(file_path, "wb") as f:
    pickle.dump(data_to_save, f)

print(f"Data successfully saved to {file_path}")

## Fine-tuning an embedding model

In [ ]:
import os
import pickle
import torch
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MarginMSELoss 
from datasets import Dataset

# --- STEP 1: FORCE SINGLE GPU ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

# --- STEP 2: PATHS AND DATA ---
dataset_path = '/mnt/c/Users/luke.flattery/Downloads/embed_model/fine_tuning_data.pkl'
gemma_path = "/mnt/c/Users/luke.flattery/Downloads/embed_model/embeddinggemma-300m"
shards = "/mnt/c/Users/luke.flattery/Downloads/embed_model/new_models/shards/"
save_path = "/mnt/c/Users/luke.flattery/Downloads/embed_model/new_models/embeddinggemma1"

with open(dataset_path, 'rb') as file:
    full_dataset = pickle.load(file)

if not isinstance(full_dataset, Dataset):
    full_dataset = Dataset.from_list(full_dataset) if isinstance(full_dataset, list) else full_dataset

# Split is still required to get "Validation Loss"
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

# --- STEP 3: MODEL AND LOSS ---
model = SentenceTransformer(gemma_path, local_files_only=True)
loss = MarginMSELoss(model)

# --- STEP 4: TRAINING ARGS ---
args = SentenceTransformerTrainingArguments(
    output_dir=shards,
    num_train_epochs=5,
    per_device_train_batch_size=5,
    learning_rate=2e-6,
    warmup_ratio=0.1,
    logging_steps=500,
    report_to="none",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=5,
)

# --- STEP 5: TRAIN ---
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss=loss,
)

print("Evaluating initial model...")
initial_metrics = trainer.evaluate()
print(f"Initial Evaluation Metrics: {initial_metrics}")

trainer.train()
trainer.save_model(save_path)

### Revert to a prior shard of the model

In [ ]:
from sentence_transformers import SentenceTransformer
import shutil
import os

# --- PATHS ---
# Update this to the exact folder name in your shards directory
checkpoint_path = "/mnt/c/Users/luke.flattery/Downloads/embed_model/new_models/shards/checkpoint-3000"
save_path = "/mnt/c/Users/luke.flattery/Downloads/embed_model/new_models/embeddinggemma1"
shards_dir = "/mnt/c/Users/luke.flattery/Downloads/embed_model/new_models/shards/"

# 1. Load the model from the specific checkpoint folder
print(f"Loading checkpoint from {checkpoint_path}...")
model = SentenceTransformer(checkpoint_path)

# 2. Save it to your final destination
# This only saves the model weights and config, excluding the optimizer/trainer state
print(f"Saving final model to {save_path}...")
model.save(save_path)

# 3. Cleanup (Optional)
# Now that you have your final model, you can delete the entire shards directory to free up space
try:
    shutil.rmtree(shards_dir)
    print("Successfully removed the shards directory.")
except Exception as e:
    print(f"Note: Could not remove shards directory: {e}")

print("Process complete.")

## General solutions for RAG problems